In [1]:
#importing the necessary libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [2]:
df = pd.read_csv('../datasets/heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [4]:
df.FastingBS.unique()

array([0, 1])

In [5]:
df.Oldpeak.unique()

array([ 0. ,  1. ,  1.5,  2. ,  3. ,  4. ,  0.5,  2.5,  5. ,  0.8,  0.7,
        1.4,  2.1,  0.4,  0.2,  1.7,  2.2,  0.1,  1.6,  1.3,  0.3,  1.8,
        2.6, -0.9,  2.8, -2.6, -1.5, -0.1,  0.9,  1.1,  2.4, -1. , -1.1,
       -0.7, -0.8,  3.7,  1.2, -0.5, -2. ,  1.9,  3.5,  0.6,  3.1,  2.3,
        3.4,  3.6,  4.2,  3.2,  5.6,  3.8,  2.9,  6.2,  4.4])

## Removing outliers using the Z score 

In [6]:
from scipy import stats

z_scores = np.abs(stats.zscore(df[['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']]))
df1 = df[(z_scores < 3).all(axis=1)].copy()


In [7]:
df1.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


## Handling text columns

In [8]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

### Sex column

In [9]:
df1.Sex.unique()

array(['M', 'F'], dtype=object)

In [10]:
encoded_sex = encoder.fit_transform(df1['Sex'])
df1['Sex'] = encoded_sex

### Chest pain type

In [11]:
df1.ChestPainType.unique()

array(['ATA', 'NAP', 'ASY', 'TA'], dtype=object)

### Resting ECG column

In [12]:
df1.RestingECG.unique()

array(['Normal', 'ST', 'LVH'], dtype=object)

### Exercise Angine column

In [13]:
df1.ExerciseAngina.unique()

array(['N', 'Y'], dtype=object)

In [14]:
encoded_exercise = encoder.fit_transform(df1['ExerciseAngina'])
df1['ExerciseAngina'] = encoded_exercise

### ST_Slope column

In [15]:
df1.ST_Slope.unique()

array(['Up', 'Flat', 'Down'], dtype=object)

In [16]:
encoded_st_slope = encoder.fit_transform(df1['ST_Slope'])
df1['ST_Slope'] = encoded_st_slope

## Using One hot encoding for RestingECG and Chest pain type

In [17]:
df1 = pd.get_dummies(df1, columns= ['RestingECG', 'ChestPainType'], dtype=int)

In [18]:
df1.head()

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA
0,40,1,140,289,0,172,0,0.0,2,0,0,1,0,0,1,0,0
1,49,0,160,180,0,156,0,1.0,1,1,0,1,0,0,0,1,0
2,37,1,130,283,0,98,0,0.0,2,0,0,0,1,0,1,0,0
3,48,0,138,214,0,108,1,1.5,1,1,0,1,0,1,0,0,0
4,54,1,150,195,0,122,0,0.0,2,0,0,1,0,0,0,1,0


### Train test split

In [19]:
from sklearn.model_selection import train_test_split, GridSearchCV

In [20]:
X = df1.drop('HeartDisease', axis='columns')
y = df1.HeartDisease

In [21]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## SVM algorithm

In [23]:
svm = GridSearchCV(
    SVC(random_state = 30),
    {
        'kernel' : ['rbf', 'linear'],
        'C': [1, 10,20],
        'gamma' : ['scale', 'auto'],
        
    },
    cv = 4,
    return_train_score=True
)
svm.fit(X_scaled,y)
print(f"The best params for the SVM model is {svm.best_params_}")
print(f"The best training accuracy score: {(svm.cv_results_['mean_train_score'][svm.best_index_] * 100):.2f}%")
print(f"The best test accuracy score: {(svm.best_score_ * 100): .2f}%")

The best params for the SVM model is {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
The best training accuracy score: 91.55%
The best test accuracy score:  83.42%


## Random Forest Algorithm

In [24]:
rf = GridSearchCV(
    RandomForestClassifier(random_state = 30),
    {
        'n_estimators': [40, 80, 120, 200, 300, 400],
        'max_depth': [3,5, 7],
        'max_features':['sqrt', 'log2'],
        'min_samples_leaf': [2,4]
        
    },
    cv = 4,
    return_train_score=True
)
rf.fit(X_scaled,y)
print(f"The best params for the Random Forest model is {rf.best_params_}")
print(f"The best training accuracy score: {(rf.cv_results_['mean_train_score'][rf.best_index_] * 100):.2f}%")
print(f" The best test accuracy score: {(rf.best_score_ * 100): .2f}%")

The best params for the Random Forest model is {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'n_estimators': 80}
The best training accuracy score: 93.99%
 The best test accuracy score:  86.42%


## Logistic Regression

In [25]:
lr = GridSearchCV(
    LogisticRegression(l1_ratio = 0, solver ='lbfgs', random_state = 30, max_iter=1000),
    {
        'C': [0.1, 1, 10, 100]
        
    },
    cv = 4,
    return_train_score=True
)
lr.fit(X_scaled,y)
print(f"The best params for the Logistic regression model is {lr.best_params_}")
print(f"The best training accuracy score: {(lr.cv_results_['mean_train_score'][lr.best_index_] * 100):.2f}%")
print(f" The best test accuracy score: {(lr.best_score_ * 100): .2f}%")

The best params for the Logistic regression model is {'C': 0.1}
The best training accuracy score: 86.02%
 The best test accuracy score:  81.97%


In [26]:
lr1 = GridSearchCV(
    LogisticRegression(l1_ratio = 1, solver ='liblinear', random_state = 30, max_iter=1000),
    {
        'C': [0.1, 1, 10, 100]
        
    },
    cv = 4,
    return_train_score=True
)
lr1.fit(X_scaled,y)
print(f"The best params for the Logistic regression model is {lr1.best_params_}")
print(f"The best training accuracy score: {(lr1.cv_results_['mean_train_score'][lr1.best_index_] * 100):.2f}%")
print(f" The best test accuracy score: {(lr1.best_score_ * 100): .2f}%")

The best params for the Logistic regression model is {'C': 1}
The best training accuracy score: 86.13%
 The best test accuracy score:  81.97%


## Applying Principal Component Analysis(PCA)

In [27]:
from sklearn.decomposition import PCA

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 30)

In [29]:
scaler = StandardScaler()
Xt_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [30]:
pca = PCA(n_components=0.95)

In [31]:
X_pca = pca.fit_transform(Xt_scaled)

In [32]:
X_test_pca = pca.transform(X_test_scaled)

## SVM Algorithm after PCA

In [33]:
svm = GridSearchCV(
    SVC(random_state = 30),
    {
        'kernel' : ['rbf', 'linear'],
        'C': [1, 10,20],
        'gamma' : ['scale', 'auto'],
        
    },
    cv = 4,
    return_train_score=False
)
svm.fit(X_pca,y_train)
print(f"The best params for the SVM model is {svm.best_params_}")
print(f"The best training accuracy score after PCA: {(svm.best_score_ * 100):.2f}%")
print(f"The best test accuracy score after PCA: {(svm.score(X_test_pca, y_test)* 100): .2f}%")

The best params for the SVM model is {'C': 1, 'gamma': 'auto', 'kernel': 'rbf'}
The best training accuracy score after PCA: 86.37%
The best test accuracy score after PCA:  85.00%


## Random Forest Algorithm after PCA

In [34]:
rf = GridSearchCV(
    RandomForestClassifier(random_state = 30),
    {
        'n_estimators': [40, 80, 120, 200, 300, 400],
        'max_depth': [3,5, 7],
        'max_features':['sqrt', 'log2'],
        'min_samples_leaf': [2,4]
        
    },
    cv = 4,
    return_train_score=False
)
rf.fit(X_pca,y_train)
print(f"The best params for the Random Forest model is {rf.best_params_}")
print(f"The best training accuracy score after PCA: {(rf.best_score_ * 100):.2f}%")
print(f" The best test accuracy score after PCA: {(rf.score(X_test_pca, y_test)* 100): .2f}%")

The best params for the Random Forest model is {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'n_estimators': 80}
The best training accuracy score after PCA: 84.84%
 The best test accuracy score after PCA:  83.89%


## Logistic Regression Algorithm after PCA

In [35]:
lr1 = GridSearchCV(
    LogisticRegression(l1_ratio = 1, solver ='liblinear', random_state = 30, max_iter=1000),
    {
        'C': [0.1, 1, 10, 100]
        
    },
    cv = 4,
    return_train_score=True
)
lr1.fit(X_pca,y_train)
print(f"The best params for the Logistic regression model is {lr1.best_params_}")
print(f"The best training accuracy score after PCA: {(lr1.best_score_ * 100):.2f}%")
print(f" The best test accuracy score after PCA: {(lr1.score(X_test_pca, y_test) * 100): .2f}%")

The best params for the Logistic regression model is {'C': 1}
The best training accuracy score after PCA: 86.23%
 The best test accuracy score after PCA:  85.00%


In [36]:
lr = GridSearchCV(
    LogisticRegression(l1_ratio = 0, solver ='lbfgs', random_state = 30, max_iter=1000),
    {
        'C': [0.1, 1, 10, 100]
        
    },
    cv = 4,
    return_train_score=True
)
lr.fit(X_pca,y_train)
print(f"The best params for the Logistic regression model is {lr.best_params_}")
print(f"The best training accuracy score after PCA: {(lr.best_score_ * 100):.2f}%")
print(f" The best test accuracy score after PCA: {(lr.score(X_test_pca, y_test) * 100): .2f}%")

The best params for the Logistic regression model is {'C': 1}
The best training accuracy score after PCA: 86.37%
 The best test accuracy score after PCA:  84.44%


# Observation
After observing the various model performances and accuracy scores, the top three models include:
1. RF without PCA - this has the best raw accuracy(86.53) but it overfits
2. LR L1 with PCA - this has the best balance (85%) with a 1.23% gap
3. SVM with PCA - this has the second best balance(85%) with a 1.37% gap

## Overall pick: is the LR L1 because it is tied for the best result and it is faster